# Balance forecasting production pipeline

Это основной запускной ноутбук. Его можно запускать целиком через `Run All`.

Куда смотреть после запуска:

1. **Run pipeline** — главный блок: здесь сразу выводится итоговое предсказание `predicted_balance` на `FORECAST_DATE`.
2. **Forecast for date** — полная таблица прогноза с фактом, ошибкой и флагами риска, если факт уже есть в данных.
3. **Model selection** и **Selected model card** — какая модель выбрана и почему.
4. **Drift and retraining decision** — есть ли сигнал разладки и нужна ли проверка/дообучение.

Исследовательские эксперименты остаются в `time_series.ipynb`.

## CONFIG

Здесь меняются только параметры запуска.

Главное поле — `forecast_date`:

- `None` означает прогноз на следующий рабочий день после последнего факта в `Project 1_2024.xlsx`;
- строка вида `"2021-03-31"` означает прогноз на конкретную дату.

Остальные параметры обычно не нужно трогать при регулярном запуске.

In [50]:
from balance_pipeline import PipelineConfig, run_pipeline

CONFIG = PipelineConfig(
    main_file="Project 1_2024.xlsx",
    macro_file="Инфляция и ключевая ставка Банка России_F01_01_2017_T29_05_2026.xlsx",
    ruonia_file="RC_F01_01_2017_T28_05_2026.xlsx",
    usd_file="RC_F01_01_2017_T30_05_2026.xlsx",
    forecast_date="2021-04-01",  # пример: "2021-03-31"; None = следующий рабочий день после последнего факта
    artifacts_dir="artifacts",
    error_threshold=0.42,
    rolling_window=500,
    cv_splits=5,
    cv_validation_size=40,
    min_train_size=180,
    allow_external_ffill=True,
    external_max_staleness_days=7,
    drift_window=20,
)
CONFIG

PipelineConfig(main_file='Project 1_2024.xlsx', macro_file='Инфляция и ключевая ставка Банка России_F01_01_2017_T29_05_2026.xlsx', ruonia_file='RC_F01_01_2017_T28_05_2026.xlsx', usd_file='RC_F01_01_2017_T30_05_2026.xlsx', forecast_date='2021-04-01', artifacts_dir='artifacts', error_threshold=0.42, rolling_window=500, cv_splits=5, cv_validation_size=40, min_train_size=180, random_state=42, allow_external_ffill=True, external_max_staleness_days=7, drift_window=20)

## Run pipeline

Это главный блок ноутбука. Он запускает весь пайплайн: загрузку данных, проверки, фичи, обучение/загрузку модели, прогноз и drift-мониторинг.

Сразу под этой ячейкой выводится короткая таблица **итогового предсказания**. Если дата уже есть в данных, рядом будет фактический `Balance` и ошибка; если даты еще нет, поля факта останутся пустыми.

In [51]:
from IPython.display import display

results = run_pipeline(CONFIG)

model_report = results["model_report"]
model_selection_report = results["model_selection_report"]
selected_model_summary = results["selected_model_summary"]
final_model = results["final_model"]
final_feature_cols = results["final_feature_cols"]
next_day_forecast = results["next_day_forecast"]
forecast_result = results["forecast_result"]
drift_report = results["drift_report"]
forecast_log = results["forecast_log"]
data_quality_report = results["data_quality_report"]

summary_cols = [
    "forecast_for_date",
    "predicted_balance",
    "actual_balance",
    "abs_error",
    "abs_error_le_0_42",
    "model_type",
    "is_tax_day_28",
    "is_tax_window_28",
    "has_recent_balance_outlier",
    "feature_outlier_alert",
    "comment",
]

print(f"FORECAST_DATE: {results['forecast_date'].date()}")
print(f"Final model: {selected_model_summary.loc[0, 'selected_model_name']} / {selected_model_summary.loc[0, 'selected_train_mode']}")
print(f"Feature count: {len(final_feature_cols)}")
print("\nИтоговое предсказание находится в таблице ниже: колонка predicted_balance.")

display(forecast_result[summary_cols])

FORECAST_DATE: 2021-04-01
Final model: ExtraTreesRegressor / all_train
Feature count: 61

Итоговое предсказание находится в таблице ниже: колонка predicted_balance.


,forecast_for_date,predicted_balance,actual_balance,abs_error,abs_error_le_0_42,model_type,is_tax_day_28,is_tax_window_28,has_recent_balance_outlier,feature_outlier_alert,comment
0,2021-04-01,-0.370752,NaN,NaN,NaN,ExtraTreesRegressor,0,0,1,True,Прогноз без факта: actual_balance и метрики ош...


## Validate data

Проверки входного файла: обязательные колонки, даты, дубликаты, пропуски, непрерывность календаря и расхождение `Balance` с `Income - Outcome`. Статус `warning` не останавливает пайплайн, но его нужно учитывать.

In [52]:
data_quality_report

,check,status,details
0,required_columns,ok,Все обязательные колонки есть
1,date_parse,ok,Некорректных дат: 0
2,duplicate_dates,ok,Дубликатов дат: 0
3,missing_values,ok,Пропусков: 0
4,time_sort,ok,Отсортировано по времени: True
5,calendar_continuity,ok,Пропущенных календарных дат: 0
6,balance_income_outcome_gap,warning,Строк с Balance != Income - Outcome при tol=1e...


## Model quality

Подробные метрики по fold-ам walk-forward проверки для бейзлайнов и ML-кандидатов.

In [53]:
model_report.sort_values(
    ["fold", "share_abs_error_le_0_42", "business_loss_mean", "MAE"],
    ascending=[True, False, True, True],
).head(30)

,fold,model_name,train_mode,feature_set,business_loss,business_loss_mean,model_profit,ideal_profit,MAE,RMSE,share_abs_error_le_0_42,bias_mean_error
8,1,Lasso,normal_train,all_features,0.000050,0.000001,-0.002399,-0.002349,0.235717,0.323697,0.900,-0.011649
5,1,rolling_mean_20,all_train,baseline,0.000050,0.000001,-0.002399,-0.002349,0.247098,0.325307,0.850,0.052858
1,1,last_value,all_train,baseline,0.000062,0.000002,-0.002411,-0.002349,0.264204,0.358209,0.825,0.012840
4,1,rolling_mean_10,all_train,baseline,0.000050,0.000001,-0.002399,-0.002349,0.246620,0.355078,0.800,0.044775
7,1,Lasso,all_train,all_features,0.000051,0.000001,-0.002400,-0.002349,0.261830,0.348473,0.800,-0.061040
10,1,ExtraTreesRegressor,normal_train,all_features,0.000050,0.000001,-0.002399,-0.002349,0.240069,0.326657,0.775,-0.098652
9,1,ExtraTreesRegressor,all_train,all_features,0.000050,0.000001,-0.002399,-0.002349,0.255495,0.334577,0.775,-0.127560
3,1,rolling_mean_5,all_train,baseline,0.000052,0.000001,-0.002401,-0.002349,0.235351,0.335720,0.775,0.029091
0,1,zero,all_train,baseline,0.000050,0.000001,-0.002399,-0.002349,0.323021,0.401600,0.725,-0.257299
6,1,day_of_week_mean,all_train,baseline,0.000058,0.000001,-0.002407,-0.002349,0.333776,0.403698,0.700,-0.246125


## Model selection

Итоговое сравнение кандидатов. Строка `selected = True` — модель, которая ушла в production-чекпоинт.

In [54]:
model_selection_report.sort_values(
    ["selected", "share_abs_error_le_0_42_rank", "business_loss_rank", "MAE_rank"],
    ascending=[False, True, True, True],
)

,model_name,train_mode,feature_set,business_loss_mean,business_loss_std,MAE_mean,RMSE_mean,share_abs_error_le_0_42_mean,bias_mean_error,business_loss_rank,MAE_rank,RMSE_rank,share_abs_error_le_0_42_rank,selected,comment
1,ExtraTreesRegressor,all_train,all_features,0.000002,0.000002,0.259788,0.360996,0.815,-0.077266,4.0,1.0,1.0,2.0,True,Выбрана по максимальной/почти максимальной дол...
0,rolling_mean_20,all_train,baseline,0.000002,0.000002,0.265161,0.365651,0.820,-0.012279,3.0,5.0,3.0,1.0,False,
2,ExtraTreesRegressor,normal_train,all_features,0.000002,0.000002,0.260914,0.365029,0.810,-0.077027,6.0,2.0,2.0,3.0,False,
3,rolling_mean_5,all_train,baseline,0.000002,0.000002,0.261682,0.371494,0.800,-0.006250,1.0,3.0,5.0,4.0,False,
4,rolling_mean_10,all_train,baseline,0.000002,0.000002,0.263493,0.368927,0.795,-0.008879,2.0,4.0,4.0,5.0,False,
5,Lasso,normal_train,all_features,0.000003,0.000002,0.292179,0.389928,0.775,-0.035177,11.0,6.0,6.0,6.0,False,
6,day_of_week_mean,all_train,baseline,0.000003,0.000002,0.298084,0.396916,0.765,-0.148046,7.0,7.0,7.0,7.0,False,
7,zero,all_train,baseline,0.000003,0.000002,0.315279,0.422312,0.755,-0.178484,8.0,9.0,9.0,8.0,False,
8,Lasso,all_train,all_features,0.000003,0.000002,0.307459,0.404635,0.740,-0.018112,10.0,8.0,8.0,9.0,False,
9,last_value,all_train,baseline,0.000003,0.000002,0.332359,0.452973,0.720,0.002717,9.0,11.0,11.0,10.0,False,


## Selected model card

Короткая карточка выбранной модели: тип, режим обучения, период train, метрики и текстовое объяснение выбора.

In [55]:
selected_model_summary

,selected_model_name,selected_train_mode,selected_feature_set,validation_strategy,business_loss_mean,MAE_mean,RMSE_mean,share_abs_error_le_0_42_mean,selection_reason,train_date_min,train_date_max
0,ExtraTreesRegressor,all_train,all_features,"walk-forward, folds=5, validation_size=40, rol...",0.000002,0.259788,0.360996,0.815,"Модель выбрана, потому что она дает лучший бал...",2017-01-09,2021-03-31


## Forecast for date

Полная таблица прогноза. Здесь больше технических полей, чем в коротком выводе после `Run pipeline`: id запуска модели, даты обучения, флаги качества данных и риска.

In [56]:
forecast_result

,forecast_created_at,forecast_for_date,predicted_balance,actual_balance,abs_error,abs_error_le_0_42,business_loss,model_run_id,model_type,train_date_min,train_date_max,is_tax_day_28,is_tax_window_28,has_recent_balance_outlier,feature_outlier_alert,target_outlier_alert,data_quality_status,comment
0,2026-05-31T21:55:00,2021-04-01,-0.370752,NaN,NaN,NaN,NaN,2026-05-31_215500_extratreesregressor,ExtraTreesRegressor,2017-01-09,2021-03-31,0,0,1,True,NaN,"warning: external_missing={'key_rate': 262, 'k...",Прогноз без факта: actual_balance и метрики ош...


## Drift and retraining decision

Отчет мониторинга после прогноза. Если факта еще нет, качество не считается; если факт есть, выводятся ошибка, rolling-метрики и рекомендация по дообучению.

In [57]:
drift_report

,forecast_for_date,has_fact,quality_drift_alert,feature_drift_alert,target_outlier_alert,drift_alert,recommendation
0,2021-04-01,False,False,True,NaN,True,Ожидать фактический Balance; мониторинг качест...


## Logs

Последние записи `forecast_log.csv`: история прогнозов, которые пайплайн уже сохранил.

In [58]:
forecast_log

,forecast_created_at,forecast_for_date,predicted_balance,actual_balance,abs_error,abs_error_le_0_42,business_loss,model_run_id,model_type,train_date_min,train_date_max,is_tax_day_28,is_tax_window_28,has_recent_balance_outlier,feature_outlier_alert,target_outlier_alert,data_quality_status,comment
0,2026-05-31T21:39:08,2021-03-31,-0.544012,-0.004878,0.539134,0.0,0.000000,2026-05-31_213908_rolling_mean_20,rolling_mean_20,2017-01-09,2021-03-30,0,0,0,True,0.0,"warning: external_missing={'key_rate': 262, 'k...",Прогноз с фактом: ошибка и бизнес-метрики расс...
1,2026-05-31T21:40:04,2021-04-01,-0.370752,NaN,NaN,NaN,NaN,2026-05-31_214004_extratreesregressor,ExtraTreesRegressor,2017-01-09,2021-03-31,0,0,1,True,NaN,"warning: external_missing={'key_rate': 262, 'k...",Прогноз без факта: actual_balance и метрики ош...
2,2026-05-31T21:41:02,2021-04-01,-0.370752,NaN,NaN,NaN,NaN,2026-05-31_214004_extratreesregressor,ExtraTreesRegressor,2017-01-09,2021-03-31,0,0,1,True,NaN,"warning: external_missing={'key_rate': 262, 'k...",Прогноз без факта: actual_balance и метрики ош...
3,2026-05-31T21:44:09,2021-01-29,-0.439813,-0.401324,0.038489,1.0,0.000000,2026-05-31_214409_rolling_mean_5,rolling_mean_5,2017-01-09,2021-01-28,0,1,0,True,0.0,"warning: external_missing={'key_rate': 262, 'k...",Прогноз с фактом: ошибка и бизнес-метрики расс...
4,2026-05-31T21:46:26,2021-01-29,-0.439813,-0.401324,0.038489,1.0,0.000000,2026-05-31_214409_rolling_mean_5,rolling_mean_5,2017-01-09,2021-01-28,0,1,0,True,0.0,"warning: external_missing={'key_rate': 262, 'k...",Прогноз с фактом: ошибка и бизнес-метрики расс...
5,2026-05-31T21:47:38,2021-01-29,-0.439813,-0.401324,0.038489,1.0,0.000000,2026-05-31_214409_rolling_mean_5,rolling_mean_5,2017-01-09,2021-01-28,0,1,0,True,0.0,"warning: external_missing={'key_rate': 262, 'k...",Прогноз с фактом: ошибка и бизнес-метрики расс...
6,2026-05-31T21:48:06,2021-03-31,-1.049968,-0.004878,1.045090,0.0,0.000000,2026-05-31_214806_rolling_mean_5,rolling_mean_5,2017-01-09,2021-03-30,0,0,0,True,0.0,"warning: external_missing={'key_rate': 262, 'k...",Прогноз с фактом: ошибка и бизнес-метрики расс...
7,2026-05-31T21:49:43,2021-03-30,-0.978423,-0.259687,0.718736,0.0,0.000000,2026-05-31_214943_rolling_mean_5,rolling_mean_5,2017-01-09,2021-03-29,0,0,1,True,0.0,"warning: external_missing={'key_rate': 262, 'k...",Прогноз с фактом: ошибка и бизнес-метрики расс...
8,2026-05-31T21:50:45,2021-04-01,-0.370752,NaN,NaN,NaN,NaN,2026-05-31_215045_extratreesregressor,ExtraTreesRegressor,2017-01-09,2021-03-31,0,0,1,True,NaN,"warning: external_missing={'key_rate': 262, 'k...",Прогноз без факта: actual_balance и метрики ош...
9,2026-05-31T21:51:28,2020-04-01,-0.109980,0.138377,0.248357,1.0,NaN,2026-05-31_215128_rolling_mean_20,rolling_mean_20,2017-01-09,2020-03-31,0,0,0,True,0.0,"warning: external_missing={'key_rate': 66, 'ke...",Прогноз с фактом: ошибка и бизнес-метрики расс...


## Final feature list

Точный список признаков и их порядок, с которыми обучена и используется финальная модель.

In [59]:
final_feature_cols

['day_of_week',
 'month',
 'is_business_month_start',
 'is_business_month_end',
 'is_business_quarter_end',
 'is_tax_day_28',
 'is_business_day_before_tax_28',
 'is_business_day_after_tax_28',
 'is_tax_window_28',
 'key_rate',
 'inflation_target',
 'key_rate_change_1m',
 'inflation_yoy_lag_1m',
 'inflation_gap_lag_1m',
 'balance_lag_1',
 'balance_lag_2',
 'balance_lag_3',
 'balance_lag_5',
 'balance_lag_10',
 'balance_lag_20',
 'balance_rolling_mean_5',
 'balance_rolling_std_5',
 'balance_rolling_mean_10',
 'balance_rolling_std_10',
 'balance_rolling_mean_20',
 'balance_rolling_std_20',
 'income_lag_1',
 'income_lag_2',
 'income_lag_3',
 'income_lag_5',
 'income_lag_10',
 'income_lag_20',
 'income_rolling_mean_5',
 'income_rolling_mean_10',
 'income_rolling_mean_20',
 'outcome_lag_1',
 'outcome_lag_2',
 'outcome_lag_3',
 'outcome_lag_5',
 'outcome_lag_10',
 'outcome_lag_20',
 'outcome_rolling_mean_5',
 'outcome_rolling_mean_10',
 'outcome_rolling_mean_20',
 'ruonia_lag_1',
 'ruonia_vol